# Code for solving the Wumpus World problem

In [1]:
import numpy as np

In [2]:
# Helper function to get the next state
def get_next_state(s, action, grid_size):
    if action == 'UP':
        return max(s - grid_size, 0) if s >= grid_size else s
    elif action == 'DOWN':
        return min(s + grid_size, grid_size**2 - 1) if s < grid_size*(grid_size - 1) else s
    elif action == 'LEFT':
        return s if s % grid_size == 0 else s - 1
    elif action == 'RIGHT':
        return s if (s + 1) % grid_size == 0 else s + 1
    return s

In [3]:
# Value Iteration Function
def MDP_value_iteration(S, A, P, R, gamma, eta, max_iter):
    U = np.zeros(len(S))

    for i in range(max_iter):
        U_prev = U.copy()

        for s in S:
            max_utility = float('-inf')
            for a in A:
                expected_utility = 0
                for s_prime, prob in P[s][a].items():
                    expected_utility += prob * (R[s_prime] + gamma * U_prev[s_prime])
                if expected_utility > max_utility:
                    max_utility = expected_utility
            U[s] = max_utility

        # Check for convergence
        if np.max(np.abs(U - U_prev)) < eta:
            break

    return U

# Policy Generation Function
def MDP_policy(S, A, P, R, U, gamma):
    policy = np.zeros(len(S), dtype=int)
    for s in S:
        max_utility = float('-inf')
        best_action = 0
        for a_idx, a in enumerate(A):
            expected_utility = 0
            for s_prime, prob in P[s][a].items():
                expected_utility += prob * (R[s_prime] + gamma * U[s_prime])
            if expected_utility > max_utility:
                max_utility = expected_utility
                best_action = a_idx
        policy[s] = best_action
    return policy

In [4]:
# Define the Wumpus world
grid_size = 4
S = range(grid_size**2)
A = ['RIGHT', 'LEFT', 'DOWN', 'UP']

# Define transition probabilities
P = {s: {a: {} for a in A} for s in S}
for s in S:
    for a in A:
        intended_state = get_next_state(s, a, grid_size)
        P[s][a][intended_state] = P[s][a].get(intended_state, 0) + 0.8
        if a in ['LEFT', 'RIGHT']:
            up_state = get_next_state(s, 'UP', grid_size)
            down_state = get_next_state(s, 'DOWN', grid_size)
            P[s][a][up_state] = P[s][a].get(up_state, 0) + 0.1
            P[s][a][down_state] = P[s][a].get(down_state, 0) + 0.1
        else:
            left_state = get_next_state(s, 'LEFT', grid_size)
            right_state = get_next_state(s, 'RIGHT', grid_size)
            P[s][a][left_state] = P[s][a].get(left_state, 0) + 0.1
            P[s][a][right_state] = P[s][a].get(right_state, 0) + 0.1

# Define rewards
R = [-0.4] * 16
R[3] = 10    # Gold
R[10] = -5   # Pit
R[14] = -5   # Pit
R[13] = -10  # Wumpus

# Run for three settings
settings = [
    (0.3, 0.1, 10000),
    (0.6, 0.1, 10000),
    (0.9, 0.1, 10000),
]

policy_repr = {0: '→', 1: '←', 2: '↓', 3: '↑'}

for gamma, eta, max_iter in settings:
    U = MDP_value_iteration(S, A, P, R, gamma, eta, max_iter)
    policy = MDP_policy(S, A, P, R, U, gamma)
    print(f"Setting: gamma={gamma}, eta={eta}, max_iter={max_iter}")
    print("Utilities and Policy for the Given Wumpus World:")
    for i in range(grid_size):
        for j in range(grid_size):
            state = i * grid_size + j
            print(f"{U[state]:.2f} {policy_repr[policy[state]]}", end=" | ")
        print()
    print()

Setting: gamma=0.3, eta=0.1, max_iter=10000
Utilities and Policy for the Given Wumpus World:
0.15 → | 2.39 → | 11.37 → | 12.71 → | 
-0.37 → | 0.26 → | 2.66 ↑ | 11.37 ↑ | 
-0.54 ↑ | -0.82 ↑ | 0.25 ↑ | 1.91 ↑ | 
-0.57 ← | -1.13 ↑ | -1.48 → | -0.48 ↑ | 

Setting: gamma=0.6, eta=0.1, max_iter=10000
Utilities and Policy for the Given Wumpus World:
4.91 → | 10.24 → | 20.30 → | 22.01 → | 
2.53 → | 5.49 → | 10.84 ↑ | 20.30 ↑ | 
0.95 ↑ | 2.11 ↑ | 5.46 ↑ | 9.74 ↑ | 
-0.79 ← | 0.11 ↑ | 0.97 → | 4.07 ↑ | 

Setting: gamma=0.9, eta=0.1, max_iter=10000
Utilities and Policy for the Given Wumpus World:
62.52 → | 72.72 → | 84.56 → | 86.77 → | 
55.76 → | 64.17 → | 73.78 ↑ | 84.56 ↑ | 
49.07 ↑ | 55.45 ↑ | 64.13 ↑ | 72.21 ↑ | 
41.94 ↑ | 47.56 ↑ | 53.36 → | 61.37 ↑ | 

